# QwenFree Kaggle server

1. In Notebook options, select **Accelerator → GPU T4 x2** and turn **Internet** on.
2. Add and attach the recommended `NGROK_AUTHTOKEN` Secret for the primary
   tunnel. `HF_TOKEN` and `QWENFREE_API_KEY` are optional.
3. Review the repository and context values in the next cell, then run all cells.
4. Copy the printed `qwenfree connect` command into Terminal on your Mac.

The notebook prefers ngrok when the secret provides a URL and falls back to
Cloudflare Quick Tunnel. Cloudflare Quick Tunnels do not support Server-Sent
Events, so QwenFree uses a completed response for `trycloudflare.com` URLs.

The final cell runs the authenticated server until you interrupt it. It reports
health every five minutes and does not try to bypass Kaggle session or quota limits.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

def optional_secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for name in ('HF_TOKEN', 'QWENFREE_API_KEY', 'NGROK_AUTHTOKEN'):
    value = optional_secret(name)
    if value:
        os.environ[name] = value

# Change this value to choose another verified Hugging Face GGUF repository.
os.environ.setdefault('MODEL_REPO', 'JonathanColetti/Qwen3.8-27B-Uncensored-GGUF')
os.environ.setdefault('CTX_SIZE', '8192')
os.environ.setdefault('MAX_MODEL_GIB', '24')
print('Secret values were loaded without displaying them.')
print('Selected model repository:', os.environ['MODEL_REPO'])


In [ ]:
from pathlib import Path
import base64

server_dir = Path('/kaggle/working/qwenfree_server')
server_dir.mkdir(parents=True, exist_ok=True)
embedded_files = {
    "bootstrap.sh": "IyEvdXNyL2Jpbi9lbnYgYmFzaApzZXQgLUVldW8gcGlwZWZhaWwKClNDUklQVF9ESVI9IiQoY2QgLS0gIiQoZGlybmFtZSAtLSAiJHtCQVNIX1NPVVJDRVswXX0iKSIgJiYgcHdkKSIKV09SS19ESVI9IiR7V09SS19ESVI6LS9rYWdnbGUvd29ya2luZ30iCkxMQU1BX0NQUF9ESVI9IiR7TExBTUFfQ1BQX0RJUjotJHtXT1JLX0RJUn0vbGxhbWEuY3BwfSIKTU9ERUxfRElSPSIke01PREVMX0RJUjotL2thZ2dsZS90ZW1wL21vZGVsfSIKQklOX0RJUj0iJHtXT1JLX0RJUn0vcXdlbmZyZWUtYmluIgpMT0dfRElSPSIke1dPUktfRElSfS9xd2VuZnJlZS1sb2dzIgpTRVJWRVJfQklOPSIke0xMQU1BX0NQUF9ESVJ9L2J1aWxkL2Jpbi9sbGFtYS1zZXJ2ZXIiCkNUWF9TSVpFPSIke0NUWF9TSVpFOi04MTkyfSIKTUFYX01PREVMX0dJQj0iJHtNQVhfTU9ERUxfR0lCOi0yNH0iCk1PREVMX1JFUE89IiR7TU9ERUxfUkVQTzotSm9uYXRoYW5Db2xldHRpL1F3ZW4zLjgtMjdCLVVuY2Vuc29yZWQtR0dVRn0iCmV4cG9ydCBNT0RFTF9SRVBPIE1BWF9NT0RFTF9HSUIKClNFUlZFUl9QSUQ9IiIKQ0xPVURGTEFSRURfUElEPSIiCk5HUk9LX1BJRD0iIgoKY2xlYW51cCgpIHsKICBsb2NhbCBleGl0X2NvZGU9JD8KICB0cmFwIC0gRVhJVCBJTlQgVEVSTQogIGVjaG8gIlN0b3BwaW5nIFF3ZW5GcmVlIHNlcnZlciBhbmQgdHVubmVsIHByb2Nlc3Nlcy4uLiIKICBmb3IgcHJvY2Vzc19pZCBpbiAiJHtDTE9VREZMQVJFRF9QSUR9IiAiJHtOR1JPS19QSUR9IiAiJHtTRVJWRVJfUElEfSI7IGRvCiAgICBpZiBbWyAtbiAiJHtwcm9jZXNzX2lkfSIgXV0gJiYga2lsbCAtMCAiJHtwcm9jZXNzX2lkfSIgMj4vZGV2L251bGw7IHRoZW4KICAgICAga2lsbCAiJHtwcm9jZXNzX2lkfSIgMj4vZGV2L251bGwgfHwgdHJ1ZQogICAgZmkKICBkb25lCiAgZm9yIHByb2Nlc3NfaWQgaW4gIiR7Q0xPVURGTEFSRURfUElEfSIgIiR7TkdST0tfUElEfSIgIiR7U0VSVkVSX1BJRH0iOyBkbwogICAgaWYgW1sgLW4gIiR7cHJvY2Vzc19pZH0iIF1dOyB0aGVuCiAgICAgIHdhaXQgIiR7cHJvY2Vzc19pZH0iIDI+L2Rldi9udWxsIHx8IHRydWUKICAgIGZpCiAgZG9uZQogIGV4aXQgIiR7ZXhpdF9jb2RlfSIKfQp0cmFwIGNsZWFudXAgRVhJVAp0cmFwICdleGl0IDEzMCcgSU5UCnRyYXAgJ2V4aXQgMTQzJyBURVJNCgpmb3IgZGVwZW5kZW5jeSBpbiBnaXQgY21ha2UgY3VybCBweXRob24zOyBkbwogIGlmICEgY29tbWFuZCAtdiAiJHtkZXBlbmRlbmN5fSIgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBlY2hvICJSZXF1aXJlZCBjb21tYW5kIG5vdCBmb3VuZDogJHtkZXBlbmRlbmN5fS4gVXNlIGEgS2FnZ2xlIEdQVSBub3RlYm9vay4iID4mMgogICAgZXhpdCAxCiAgZmkKZG9uZQoKaWYgISBjb21tYW5kIC12IG52aWRpYS1zbWkgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgZWNobyAibnZpZGlhLXNtaSBpcyBtaXNzaW5nOyBlbmFibGUgQWNjZWxlcmF0b3IgLT4gR1BVIFQ0IHgyIGluIEthZ2dsZSBzZXR0aW5ncy4iID4mMgogIGV4aXQgMQpmaQpHUFVfQ09VTlQ9IiQobnZpZGlhLXNtaSAtTCB8IGdyZXAgLWMgJ15HUFUgJyB8fCB0cnVlKSIKaWYgW1sgIiR7R1BVX0NPVU5UfSIgIT0gIjIiIF1dOyB0aGVuCiAgZWNobyAiRXhwZWN0ZWQgZXhhY3RseSAyIEdQVXMsIGZvdW5kICR7R1BVX0NPVU5UfS4gU2VsZWN0IEdQVSBUNCB4MiBhbmQgcmVydW4uIiA+JjIKICBudmlkaWEtc21pIC1MIHx8IHRydWUKICBleGl0IDEKZmkKZWNobyAiRGV0ZWN0ZWQgdHdvIEdQVXM6IgpudmlkaWEtc21pIC0tcXVlcnktZ3B1PW5hbWUsbWVtb3J5LnRvdGFsIC0tZm9ybWF0PWNzdixub2hlYWRlcgoKaWYgW1sgISAiJHtDVFhfU0laRX0iID1+IF5bMC05XSskIF1dIHx8ICgoIENUWF9TSVpFIDwgMTAyNCB8fCBDVFhfU0laRSA+IDMyNzY4ICkpOyB0aGVuCiAgZWNobyAiQ1RYX1NJWkUgbXVzdCBiZSBhbiBpbnRlZ2VyIGZyb20gMTAyNCB0byAzMjc2ODsgbmF0aXZlIDI2MmsgY29udGV4dCBpcyBub3QgdXNlZC4iID4mMgogIGV4aXQgMQpmaQoKbWtkaXIgLXAgIiR7V09SS19ESVJ9IiAiJHtCSU5fRElSfSIgIiR7TE9HX0RJUn0iICIke01PREVMX0RJUn0iCmlmICEgcHl0aG9uMyAtYyAnaW1wb3J0IGh1Z2dpbmdmYWNlX2h1YicgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgZWNobyAiSW5zdGFsbGluZyB0aGUgc21hbGwgc2VydmVyLXNpZGUgSHVnZ2luZyBGYWNlIGNsaWVudC4uLiIKICBweXRob24zIC1tIHBpcCBpbnN0YWxsIC0tcXVpZXQgaHVnZ2luZ2ZhY2VfaHViCmZpCgppZiBbWyAtZCAiJHtMTEFNQV9DUFBfRElSfS8uZ2l0IiBdXTsgdGhlbgogIGVjaG8gIlVwZGF0aW5nIGxsYW1hLmNwcCBmcm9tIGl0cyBjb25maWd1cmVkIGRlZmF1bHQgYnJhbmNoLi4uIgogIGdpdCAtQyAiJHtMTEFNQV9DUFBfRElSfSIgcHVsbCAtLWZmLW9ubHkKZWxzZQogIGVjaG8gIkNsb25pbmcgdGhlIGxhdGVzdCBkZWZhdWx0IGJyYW5jaCBvZiBnZ21sLW9yZy9sbGFtYS5jcHAuLi4iCiAgZ2l0IGNsb25lIC0tZGVwdGggMSBodHRwczovL2dpdGh1Yi5jb20vZ2dtbC1vcmcvbGxhbWEuY3BwLmdpdCAiJHtMTEFNQV9DUFBfRElSfSIKZmkKCmVjaG8gIkJ1aWxkaW5nIGxsYW1hLXNlcnZlciB3aXRoIENVREEgZm9yIE5WSURJQSBUNCAoY29tcHV0ZSBjYXBhYmlsaXR5IDcuNSkuLi4iCmlmIFtbIC1kIC91c3IvbG9jYWwvbnZpZGlhL2xpYjY0IF1dOyB0aGVuCiAgZXhwb3J0IENNQUtFX0xJQlJBUllfUEFUSD0iL3Vzci9sb2NhbC9udmlkaWEvbGliNjQke0NNQUtFX0xJQlJBUllfUEFUSDorOiR7Q01BS0VfTElCUkFSWV9QQVRIfX0iCiAgZWNobyAiSW5jbHVkaW5nIEthZ2dsZSdzIE5WSURJQSBkcml2ZXIgbGlicmFyaWVzIGluIENNYWtlJ3Mgc2VhcmNoIHBhdGguIgpmaQpjbWFrZSAtUyAiJHtMTEFNQV9DUFBfRElSfSIgLUIgIiR7TExBTUFfQ1BQX0RJUn0vYnVpbGQiIFwKICAtREdHTUxfQ1VEQT1PTiBcCiAgLURDTUFLRV9DVURBX0FSQ0hJVEVDVFVSRVM9NzUgXAogIC1EQ01BS0VfQlVJTERfVFlQRT1SZWxlYXNlCmNtYWtlIC0tYnVpbGQgIiR7TExBTUFfQ1BQX0RJUn0vYnVpbGQiIC0tY29uZmlnIFJlbGVhc2UgXAogIC0tdGFyZ2V0IGxsYW1hLXNlcnZlciAtaiA0CgppZiBbWyAhIC14ICIke1NFUlZFUl9CSU59IiBdXTsgdGhlbgogIGVjaG8gIkJ1aWxkIGNvbXBsZXRlZCwgYnV0IGxsYW1hLXNlcnZlciB3YXMgbm90IGZvdW5kIGF0ICR7U0VSVkVSX0JJTn0uIiA+JjIKICBleGl0IDEKZmkKClNFTEVDVElPTl9GSUxFPSIke1dPUktfRElSfS9xd2VuZnJlZS1tb2RlbC1zZWxlY3Rpb24uanNvbiIKZWNobyAiQ2hlY2tpbmcgdGhlIHNlbGVjdGVkIEh1Z2dpbmcgRmFjZSByZXBvc2l0b3J5IGFuZCBHR1VGIHNpemVzLi4uIgpweXRob24zICIke1NDUklQVF9ESVJ9L2ZpbmRfbW9kZWwucHkiIC0tanNvbiAtLW1heC1tb2RlbC1naWIgIiR7TUFYX01PREVMX0dJQn0iIFwKICA+ICIke1NFTEVDVElPTl9GSUxFfSIKCnB5dGhvbjMgLSAiJHtTRUxFQ1RJT05fRklMRX0iICIke01PREVMX0RJUn0iIDw8J1BZJwppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN5cwpmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgc25hcHNob3RfZG93bmxvYWQKCnNlbGVjdGlvbl9wYXRoLCBtb2RlbF9kaXIgPSBzeXMuYXJndlsxOl0Kd2l0aCBvcGVuKHNlbGVjdGlvbl9wYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBzb3VyY2U6CiAgICBzZWxlY3Rpb24gPSBqc29uLmxvYWQoc291cmNlKQpwcmludCgKICAgICJEb3dubG9hZGluZyB7fSB7fSAoezouMmZ9IEdpQiBvZiB3ZWlnaHRzKSBmcm9tIHt9Ii5mb3JtYXQoCiAgICAgICAgc2VsZWN0aW9uWyJxdWFudCJdLAogICAgICAgIHNlbGVjdGlvblsicHJpbWFyeV9maWxlIl0sCiAgICAgICAgc2VsZWN0aW9uWyJ0b3RhbF9zaXplX2J5dGVzIl0gLyAxMDI0KiozLAogICAgICAgIHNlbGVjdGlvblsicmVwb19pZCJdLAogICAgKQopCnNuYXBzaG90X2Rvd25sb2FkKAogICAgcmVwb19pZD1zZWxlY3Rpb25bInJlcG9faWQiXSwKICAgIGxvY2FsX2Rpcj1tb2RlbF9kaXIsCiAgICBhbGxvd19wYXR0ZXJucz1zZWxlY3Rpb25bImZpbGVzIl0sCiAgICB0b2tlbj1vcy5lbnZpcm9uLmdldCgiSEZfVE9LRU4iKSBvciBOb25lLAopCmZvciBmaWxlbmFtZSBpbiBzZWxlY3Rpb25bImZpbGVzIl06CiAgICBpZiBub3Qgb3MucGF0aC5pc2ZpbGUob3MucGF0aC5qb2luKG1vZGVsX2RpciwgZmlsZW5hbWUpKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCJNb2RlbCBkb3dubG9hZCBpcyBpbmNvbXBsZXRlOiAiICsgZmlsZW5hbWUpClBZCgpQUklNQVJZX0ZJTEU9IiQocHl0aG9uMyAtYyAnaW1wb3J0IGpzb24sc3lzOyBwcmludChqc29uLmxvYWQob3BlbihzeXMuYXJndlsxXSkpWyJwcmltYXJ5X2ZpbGUiXSknICIke1NFTEVDVElPTl9GSUxFfSIpIgpNT0RFTF9QQVRIPSIke01PREVMX0RJUn0vJHtQUklNQVJZX0ZJTEV9IgppZiBbWyAhIC1zICIke01PREVMX1BBVEh9IiBdXTsgdGhlbgogIGVjaG8gIlNlbGVjdGVkIG1vZGVsIGZpbGUgaXMgbWlzc2luZyBvciBlbXB0eTogJHtNT0RFTF9QQVRIfSIgPiYyCiAgZXhpdCAxCmZpCgpTRVJWRVJfSEVMUD0iJCgiJHtTRVJWRVJfQklOfSIgLS1oZWxwIDI+JjEgfHwgdHJ1ZSkiCnJlcXVpcmVfc2VydmVyX29wdGlvbigpIHsKICBsb2NhbCBvcHRpb249IiQxIgogIGlmICEgZ3JlcCAtRnEgLS0gIiR7b3B0aW9ufSIgPDw8IiR7U0VSVkVSX0hFTFB9IjsgdGhlbgogICAgZWNobyAiVGhlIGJ1aWx0IGxsYW1hLXNlcnZlciBkb2VzIG5vdCBzdXBwb3J0IHJlcXVpcmVkIG9wdGlvbiAke29wdGlvbn0uIiA+JjIKICAgIGVjaG8gIlJlYnVpbGQgZnJvbSB0aGUgbGF0ZXN0IGxsYW1hLmNwcCBkZWZhdWx0IGJyYW5jaCBhbmQgdHJ5IGFnYWluLiIgPiYyCiAgICBleGl0IDEKICBmaQp9CmZvciBvcHRpb24gaW4gLS10ZW5zb3Itc3BsaXQgLS1mbGFzaC1hdHRuIC0tY3R4LXNpemUgLS1ob3N0IC0tcG9ydCAtLWFwaS1rZXkgLS1qaW5qYTsgZG8KICByZXF1aXJlX3NlcnZlcl9vcHRpb24gIiR7b3B0aW9ufSIKZG9uZQoKU0VSVkVSX0FSR1M9KAogIC1tICIke01PREVMX1BBVEh9IgogIC1uZ2wgOTkKICAtLXRlbnNvci1zcGxpdCAxLDEKICAtYyAiJHtDVFhfU0laRX0iCiAgLS1ob3N0IDEyNy4wLjAuMQogIC0tcG9ydCA4MDgwCiAgLS1qaW5qYQopCmlmIGdyZXAgLUZxIC0tICctLWZsYXNoLWF0dG4gW29ufG9mZnxhdXRvXScgPDw8IiR7U0VSVkVSX0hFTFB9IiB8fCBcCiAgZ3JlcCAtRnEgLS0gJy0tZmxhc2gtYXR0biA8b258b2ZmfGF1dG8+JyA8PDwiJHtTRVJWRVJfSEVMUH0iOyB0aGVuCiAgU0VSVkVSX0FSR1MrPSgtLWZsYXNoLWF0dG4gb24pCmVsc2UKICBTRVJWRVJfQVJHUys9KC0tZmxhc2gtYXR0bikKZmkKaWYgZ3JlcCAtRnEgLS0gJy0tY2FjaGUtdHlwZS1rJyA8PDwiJHtTRVJWRVJfSEVMUH0iICYmIFwKICBncmVwIC1GcSAtLSAnLS1jYWNoZS10eXBlLXYnIDw8PCIke1NFUlZFUl9IRUxQfSIgJiYgXAogIGdyZXAgLUZxICdxOF8wJyA8PDwiJHtTRVJWRVJfSEVMUH0iOyB0aGVuCiAgU0VSVkVSX0FSR1MrPSgtLWNhY2hlLXR5cGUtayBxOF8wIC0tY2FjaGUtdHlwZS12IHE4XzApCiAgZWNobyAiVXNpbmcgcThfMCBLViBjYWNoZSBiZWNhdXNlIHRoaXMgYnVpbGQgYWR2ZXJ0aXNlcyBib3RoIGNhY2hlIGZsYWdzLiIKZWxzZQogIGVjaG8gIlRoaXMgbGxhbWEtc2VydmVyIGJ1aWxkIGRvZXMgbm90IGFkdmVydGlzZSBxOF8wIEtWIGNhY2hlOyB1c2luZyBpdHMgZGVmYXVsdCBLViBjYWNoZS4iCmZpCmlmIGdyZXAgLUZxIC0tICctLXBhcmFsbGVsJyA8PDwiJHtTRVJWRVJfSEVMUH0iOyB0aGVuCiAgU0VSVkVSX0FSR1MrPSgtLXBhcmFsbGVsIDEpCmZpCgppZiBbWyAtbiAiJHtRV0VORlJFRV9BUElfS0VZOi19IiBdXTsgdGhlbgogIEFQSV9LRVk9IiR7UVdFTkZSRUVfQVBJX0tFWX0iCmVsc2UKICBBUElfS0VZPSIkKHB5dGhvbjMgLWMgJ2ltcG9ydCBzZWNyZXRzOyBwcmludChzZWNyZXRzLnRva2VuX3VybHNhZmUoMzIpKScpIgpmaQoKZWNobyAiU3RhcnRpbmcgbGxhbWEtc2VydmVyIG9uIGxvb3BiYWNrLi4uIgoiJHtTRVJWRVJfQklOfSIgIiR7U0VSVkVSX0FSR1NbQF19IiAtLWFwaS1rZXkgIiR7QVBJX0tFWX0iIFwKICA+ICIke0xPR19ESVJ9L2xsYW1hLXNlcnZlci5sb2ciIDI+JjEgJgpTRVJWRVJfUElEPSQhCgpTRVJWRVJfUkVBRFk9ImZhbHNlIgpmb3IgXyBpbiAkKHNlcSAxIDE4MCk7IGRvCiAgaWYgISBraWxsIC0wICIke1NFUlZFUl9QSUR9IiAyPi9kZXYvbnVsbDsgdGhlbgogICAgYnJlYWsKICBmaQogIGlmIGN1cmwgLS1zaWxlbnQgLS1zaG93LWVycm9yIC0tZmFpbCAtLW1heC10aW1lIDUgXAogICAgaHR0cDovLzEyNy4wLjAuMTo4MDgwL2hlYWx0aCA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICAgIFNFUlZFUl9SRUFEWT0idHJ1ZSIKICAgIGJyZWFrCiAgZmkKICBzbGVlcCA1CmRvbmUKaWYgW1sgIiR7U0VSVkVSX1JFQURZfSIgIT0gInRydWUiIF1dOyB0aGVuCiAgaWYgZ3JlcCAtRXFpICd1bmtub3duIChtb2RlbCApP2FyY2hpdGVjdHVyZScgIiR7TE9HX0RJUn0vbGxhbWEtc2VydmVyLmxvZyI7IHRoZW4KICAgIGVjaG8gIlVua25vd24gbW9kZWwgYXJjaGl0ZWN0dXJlOiByZWJ1aWxkIGxsYW1hLmNwcCBmcm9tIGl0cyBsYXRlc3QgbWFzdGVyL2RlZmF1bHQgYnJhbmNoLiIgPiYyCiAgZWxzZQogICAgZWNobyAibGxhbWEtc2VydmVyIGRpZCBub3QgYmVjb21lIGhlYWx0aHkuIFJlY2VudCBzZXJ2ZXIgbG9nIGZvbGxvd3M6IiA+JjIKICBmaQogIHRhaWwgLW4gNjAgIiR7TE9HX0RJUn0vbGxhbWEtc2VydmVyLmxvZyIgPiYyIHx8IHRydWUKICBleGl0IDEKZmkKCkNMT1VERkxBUkVEX0JJTj0iJHtCSU5fRElSfS9jbG91ZGZsYXJlZCIKaWYgW1sgISAteCAiJHtDTE9VREZMQVJFRF9CSU59IiBdXTsgdGhlbgogIGVjaG8gIkRvd25sb2FkaW5nIGNsb3VkZmxhcmVkIGxpbnV4LWFtZDY0IGZyb20gQ2xvdWRmbGFyZSdzIG9mZmljaWFsIEdpdEh1YiByZWxlYXNlLi4uIgogIGN1cmwgLS1mYWlsIC0tbG9jYXRpb24gLS1zaWxlbnQgLS1zaG93LWVycm9yIFwKICAgIGh0dHBzOi8vZ2l0aHViLmNvbS9jbG91ZGZsYXJlL2Nsb3VkZmxhcmVkL3JlbGVhc2VzL2xhdGVzdC9kb3dubG9hZC9jbG91ZGZsYXJlZC1saW51eC1hbWQ2NCBcCiAgICAtLW91dHB1dCAiJHtDTE9VREZMQVJFRF9CSU59IgogIGNobW9kIDcwMCAiJHtDTE9VREZMQVJFRF9CSU59IgpmaQoiJHtDTE9VREZMQVJFRF9CSU59IiAtLXZlcnNpb24KIiR7Q0xPVURGTEFSRURfQklOfSIgdHVubmVsIC0tdXJsIGh0dHA6Ly8xMjcuMC4wLjE6ODA4MCBcCiAgPiAiJHtMT0dfRElSfS9jbG91ZGZsYXJlZC5sb2ciIDI+JjEgJgpDTE9VREZMQVJFRF9QSUQ9JCEKClBVQkxJQ19VUkw9IiIKZm9yIF8gaW4gJChzZXEgMSAzMCk7IGRvCiAgUFVCTElDX1VSTD0iJChncmVwIC1FbyAnaHR0cHM6Ly9bWzphbG51bTpdLV0rXC50cnljbG91ZGZsYXJlXC5jb20nIFwKICAgICIke0xPR19ESVJ9L2Nsb3VkZmxhcmVkLmxvZyIgfCBoZWFkIC1uIDEgfHwgdHJ1ZSkiCiAgaWYgW1sgLW4gIiR7UFVCTElDX1VSTH0iIF1dOyB0aGVuCiAgICBicmVhawogIGZpCiAgaWYgISBraWxsIC0wICIke0NMT1VERkxBUkVEX1BJRH0iIDI+L2Rldi9udWxsOyB0aGVuCiAgICBicmVhawogIGZpCiAgc2xlZXAgMgpkb25lCgppZiBbWyAtbiAiJHtOR1JPS19BVVRIVE9LRU46LX0iIF1dOyB0aGVuCiAgaWYgW1sgLW4gIiR7UFVCTElDX1VSTH0iIF1dOyB0aGVuCiAgICBlY2hvICJOR1JPS19BVVRIVE9LRU4gaXMgY29uZmlndXJlZDsgdHJ5aW5nIG5ncm9rIGZvciB0b2tlbiBzdHJlYW1pbmcuIgogICAgZWNobyAiQ2xvdWRmbGFyZSBRdWljayBUdW5uZWxzIGRvIG5vdCBzdXBwb3J0IFNlcnZlci1TZW50IEV2ZW50cy4iCiAgZWxzZQogICAgZWNobyAiQ2xvdWRmbGFyZSBxdWljayB0dW5uZWwgZGlkIG5vdCBwcm92aWRlIGEgVVJMOyB0cnlpbmcgb3B0aW9uYWwgbmdyb2sgZmFsbGJhY2suLi4iCiAgZmkKICBOR1JPS19ESVI9IiR7QklOX0RJUn0vbmdyb2siCiAgbWtkaXIgLXAgIiR7TkdST0tfRElSfSIKICBpZiBbWyAhIC14ICIke05HUk9LX0RJUn0vbmdyb2siIF1dOyB0aGVuCiAgICBjdXJsIC0tZmFpbCAtLWxvY2F0aW9uIC0tc2lsZW50IC0tc2hvdy1lcnJvciBcCiAgICAgIGh0dHBzOi8vYmluLm5ncm9rLmNvbS9jL2JOeWoxbVFWWTRjL25ncm9rLXYzLXN0YWJsZS1saW51eC1hbWQ2NC50Z3ogXAogICAgICAtLW91dHB1dCAiJHtOR1JPS19ESVJ9L25ncm9rLnRneiIKICAgIHRhciAteHpmICIke05HUk9LX0RJUn0vbmdyb2sudGd6IiAtQyAiJHtOR1JPS19ESVJ9IgogICAgY2htb2QgNzAwICIke05HUk9LX0RJUn0vbmdyb2siCiAgZmkKICBOR1JPS19DT05GSUc9IiR7V09SS19ESVJ9L3F3ZW5mcmVlLW5ncm9rLnltbCIKICAiJHtOR1JPS19ESVJ9L25ncm9rIiBjb25maWcgYWRkLWF1dGh0b2tlbiAiJHtOR1JPS19BVVRIVE9LRU59IiBcCiAgICAtLWNvbmZpZyAiJHtOR1JPS19DT05GSUd9IiA+L2Rldi9udWxsCiAgY2htb2QgNjAwICIke05HUk9LX0NPTkZJR30iCiAgIiR7TkdST0tfRElSfS9uZ3JvayIgaHR0cCA4MDgwIC0tY29uZmlnICIke05HUk9LX0NPTkZJR30iIFwKICAgIC0tbG9nIHN0ZG91dCAtLWxvZy1mb3JtYXQganNvbiA+ICIke0xPR19ESVJ9L25ncm9rLmxvZyIgMj4mMSAmCiAgTkdST0tfUElEPSQhCiAgTkdST0tfVVJMPSIiCiAgZm9yIF8gaW4gJChzZXEgMSAzMCk7IGRvCiAgICBOR1JPS19VUkw9IiQocHl0aG9uMyAtIDw8J1BZJwppbXBvcnQganNvbgppbXBvcnQgdXJsbGliLmVycm9yCmltcG9ydCB1cmxsaWIucmVxdWVzdAoKdHJ5OgogICAgd2l0aCB1cmxsaWIucmVxdWVzdC51cmxvcGVuKCJodHRwOi8vMTI3LjAuMC4xOjQwNDAvYXBpL3R1bm5lbHMiLCB0aW1lb3V0PTEpIGFzIHJlc3BvbnNlOgogICAgICAgIHR1bm5lbHMgPSBqc29uLmxvYWQocmVzcG9uc2UpLmdldCgidHVubmVscyIsIFtdKQogICAgcHJpbnQobmV4dCgoaXRlbVsicHVibGljX3VybCJdIGZvciBpdGVtIGluIHR1bm5lbHMgaWYgaXRlbS5nZXQoInByb3RvIikgPT0gImh0dHBzIiksICIiKSkKZXhjZXB0IChPU0Vycm9yLCB1cmxsaWIuZXJyb3IuVVJMRXJyb3IsIGpzb24uSlNPTkRlY29kZUVycm9yKToKICAgIHByaW50KCIiKQpQWQogICAgKSIKICAgIGlmIFtbIC1uICIke05HUk9LX1VSTH0iIF1dOyB0aGVuCiAgICAgIGJyZWFrCiAgICBmaQogICAgc2xlZXAgMgogIGRvbmUKICBpZiBbWyAtbiAiJHtOR1JPS19VUkx9IiBdXTsgdGhlbgogICAgUFVCTElDX1VSTD0iJHtOR1JPS19VUkx9IgogICAgaWYgW1sgLW4gIiR7Q0xPVURGTEFSRURfUElEfSIgXV0gJiYga2lsbCAtMCAiJHtDTE9VREZMQVJFRF9QSUR9IiAyPi9kZXYvbnVsbDsgdGhlbgogICAgICBraWxsICIke0NMT1VERkxBUkVEX1BJRH0iIDI+L2Rldi9udWxsIHx8IHRydWUKICAgICAgd2FpdCAiJHtDTE9VREZMQVJFRF9QSUR9IiAyPi9kZXYvbnVsbCB8fCB0cnVlCiAgICAgIENMT1VERkxBUkVEX1BJRD0iIgogICAgZmkKICBlbGlmIFtbIC16ICIke1BVQkxJQ19VUkx9IiBdXTsgdGhlbgogICAgZWNobyAibmdyb2sgZGlkIG5vdCBwcm92aWRlIGEgVVJMOyBjaGVjayAke0xPR19ESVJ9L25ncm9rLmxvZy4iID4mMgogIGZpCmZpCgppZiBbWyAteiAiJHtQVUJMSUNfVVJMfSIgXV07IHRoZW4KICBlY2hvICJObyBwdWJsaWMgdHVubmVsIFVSTCB3YXMgb2J0YWluZWQuIFJlY2VudCB0dW5uZWwgbG9nIGZvbGxvd3M6IiA+JjIKICB0YWlsIC1uIDQwICIke0xPR19ESVJ9L2Nsb3VkZmxhcmVkLmxvZyIgPiYyIHx8IHRydWUKICBpZiBbWyAteiAiJHtOR1JPS19BVVRIVE9LRU46LX0iIF1dOyB0aGVuCiAgICBlY2hvICJTZXQgdGhlIG9wdGlvbmFsIE5HUk9LX0FVVEhUT0tFTiBLYWdnbGUgU2VjcmV0IHRvIGVuYWJsZSBuZ3JvayBmYWxsYmFjay4iID4mMgogIGZpCiAgZXhpdCAxCmZpCgpjYXQgPDxFT0YKCj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpRd2VuRnJlZSBzZXJ2ZXIgaXMgcmVhZHkuIEtlZXAgdGhpcyBLYWdnbGUgY2VsbCBydW5uaW5nLgpRV0VORlJFRV9VUkw9JHtQVUJMSUNfVVJMfQpRV0VORlJFRV9LRVk9JHtBUElfS0VZfQoKT24geW91ciBNYWMsIHJ1bjoKcXdlbmZyZWUgY29ubmVjdCAke1BVQkxJQ19VUkx9IC0ta2V5ICR7QVBJX0tFWX0KPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkVPRgoKd2hpbGUga2lsbCAtMCAiJHtTRVJWRVJfUElEfSIgMj4vZGV2L251bGw7IGRvCiAgc2xlZXAgMzAwCiAgaWYgY3VybCAtLXNpbGVudCAtLXNob3ctZXJyb3IgLS1mYWlsIC0tbWF4LXRpbWUgMTAgXAogICAgaHR0cDovLzEyNy4wLjAuMTo4MDgwL2hlYWx0aCA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICAgIGVjaG8gIiQoZGF0ZSAtdSAnKyVZLSVtLSVkICVIOiVNOiVTIFVUQycpIFF3ZW5GcmVlIHNlcnZlciBoZWFsdGg6IE9LIgogIGVsc2UKICAgIGVjaG8gIiQoZGF0ZSAtdSAnKyVZLSVtLSVkICVIOiVNOiVTIFVUQycpIFF3ZW5GcmVlIHNlcnZlciBoZWFsdGg6IHVuYXZhaWxhYmxlIgogIGZpCmRvbmUKCmVjaG8gImxsYW1hLXNlcnZlciBzdG9wcGVkOyBjaGVjayAke0xPR19ESVJ9L2xsYW1hLXNlcnZlci5sb2cuIgo=",
    "find_model.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJTZWxlY3QgYSB2ZXJpZmllZCBRd2VuMy44LTI3QiB1bmNlbnNvcmVkIEdHVUYgZnJvbSBIdWdnaW5nIEZhY2UuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc3lzCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGFzZGljdCwgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUHVyZVBvc2l4UGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgpERUZBVUxUX1JFUE8gPSAiSm9uYXRoYW5Db2xldHRpL1F3ZW4zLjgtMjdCLVVuY2Vuc29yZWQtR0dVRiIKREVGQVVMVF9NQVhfTU9ERUxfR0lCID0gMjQuMApRVUFOVF9PUkRFUiA9ICgiUTRfS19NIiwgIklRNF9YUyIsICJRM19LX00iKQpHSUIgPSAxMDI0KiozCgoKY2xhc3MgTW9kZWxTZWxlY3Rpb25FcnJvcihSdW50aW1lRXJyb3IpOgogICAgIiIiUmFpc2VkIHdoZW4gbm8gdmVyaWZpZWQgR0dVRiBmaXRzIHRoZSBjb25maWd1cmVkIG1lbW9yeSBidWRnZXQuIiIiCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSwgc2xvdHM9VHJ1ZSkKY2xhc3MgTW9kZWxTZWxlY3Rpb246CiAgICAiIiJUaGUgZXhhY3QgZmlsZXMgYW5kIG1vZGVsIG1ldGFkYXRhIG5lZWRlZCBieSB0aGUgYm9vdHN0cmFwIHNjcmlwdC4iIiIKCiAgICByZXBvX2lkOiBzdHIKICAgIHF1YW50OiBzdHIKICAgIGZpbGVzOiBsaXN0W3N0cl0KICAgIHByaW1hcnlfZmlsZTogc3RyCiAgICB0b3RhbF9zaXplX2J5dGVzOiBpbnQKICAgIG1heF9tb2RlbF9naWI6IGZsb2F0CgoKZGVmIF9nZXQoaXRlbTogQW55LCBrZXk6IHN0ciwgZGVmYXVsdDogQW55ID0gTm9uZSkgLT4gQW55OgogICAgIiIiUmVhZCBhIGZpZWxkIGZyb20gZWl0aGVyIEhGIG1ldGFkYXRhIG9iamVjdHMgb3IgdGVzdCBkaWN0aW9uYXJpZXMuIiIiCiAgICBpZiBpc2luc3RhbmNlKGl0ZW0sIGRpY3QpOgogICAgICAgIHJldHVybiBpdGVtLmdldChrZXksIGRlZmF1bHQpCiAgICByZXR1cm4gZ2V0YXR0cihpdGVtLCBrZXksIGRlZmF1bHQpCgoKZGVmIHF1YW50X2Zvcl9maWxlbmFtZShmaWxlbmFtZTogc3RyKSAtPiBzdHIgfCBOb25lOgogICAgIiIiUmV0dXJuIGEgc3VwcG9ydGVkIHF1YW50IG5hbWUgd2hlbiBwcmVzZW50IGluIGEgR0dVRiBmaWxlbmFtZS4iIiIKICAgIGlmIG5vdCBmaWxlbmFtZS5sb3dlcigpLmVuZHN3aXRoKCIuZ2d1ZiIpOgogICAgICAgIHJldHVybiBOb25lCiAgICBsb3dlcmVkID0gZmlsZW5hbWUubG93ZXIoKQogICAgcGF0dGVybnMgPSB7CiAgICAgICAgIlE0X0tfTSI6IHIiKD88IVthLXowLTldKXE0W18tXT9rW18tXT9tKD8hW2EtejAtOV0pIiwKICAgICAgICAiSVE0X1hTIjogciIoPzwhW2EtejAtOV0paXE0W18tXT94cyg/IVthLXowLTldKSIsCiAgICAgICAgIlEzX0tfTSI6IHIiKD88IVthLXowLTldKXEzW18tXT9rW18tXT9tKD8hW2EtejAtOV0pIiwKICAgIH0KICAgIGZvciBxdWFudCwgcGF0dGVybiBpbiBwYXR0ZXJucy5pdGVtcygpOgogICAgICAgIGlmIHJlLnNlYXJjaChwYXR0ZXJuLCBsb3dlcmVkKToKICAgICAgICAgICAgcmV0dXJuIHF1YW50CiAgICByZXR1cm4gTm9uZQoKCmRlZiBfc2FmZV9yZXBvX2ZpbGVuYW1lKGZpbGVuYW1lOiBzdHIpIC0+IGJvb2w6CiAgICAiIiJSZWplY3QgYWJzb2x1dGUgcGF0aHMgYW5kIHRyYXZlcnNhbCBpbiB1bnRydXN0ZWQgSHViIGZpbGUgbWV0YWRhdGEuIiIiCiAgICBwYXRoID0gUHVyZVBvc2l4UGF0aChmaWxlbmFtZSkKICAgIHJldHVybiBib29sKGZpbGVuYW1lKSBhbmQgbm90IHBhdGguaXNfYWJzb2x1dGUoKSBhbmQgIlxcIiBub3QgaW4gZmlsZW5hbWUgYW5kIGFsbCgKICAgICAgICBwYXJ0IG5vdCBpbiB7IiIsICIuIiwgIi4uIn0gZm9yIHBhcnQgaW4gcGF0aC5wYXJ0cwogICAgKQoKCmRlZiBfZ3JvdXBzX2Zvcl9xdWFudChzaWJsaW5nczogbGlzdFtBbnldLCBxdWFudDogc3RyKSAtPiBsaXN0W3R1cGxlW2xpc3Rbc3RyXSwgaW50XV06CiAgICAiIiJHcm91cCBhIHF1YW50J3MgZmlsZXMgaW50byBzaW5nbGUgR0dVRnMgb3IgY29tcGxldGUgc2hhcmQgc2V0cy4iIiIKICAgIGdyb3VwczogZGljdFtzdHIsIGRpY3Rbc3RyLCBBbnldXSA9IHt9CiAgICBmb3Igc2libGluZyBpbiBzaWJsaW5nczoKICAgICAgICBmaWxlbmFtZSA9IHN0cihfZ2V0KHNpYmxpbmcsICJyZmlsZW5hbWUiLCAiIikpCiAgICAgICAgc2l6ZSA9IF9nZXQoc2libGluZywgInNpemUiKQogICAgICAgIGlmICgKICAgICAgICAgICAgbm90IF9zYWZlX3JlcG9fZmlsZW5hbWUoZmlsZW5hbWUpCiAgICAgICAgICAgIG9yIHF1YW50X2Zvcl9maWxlbmFtZShmaWxlbmFtZSkgIT0gcXVhbnQKICAgICAgICAgICAgb3Igbm90IGlzaW5zdGFuY2Uoc2l6ZSwgaW50KQogICAgICAgICAgICBvciBzaXplIDw9IDAKICAgICAgICApOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHNwbGl0ID0gcmUubWF0Y2goCiAgICAgICAgICAgIHIiXig/UDxzdGVtPi4qKS0oP1A8aW5kZXg+XGR7NX0pLW9mLSg/UDxjb3VudD5cZHs1fSlcLmdndWYkIiwKICAgICAgICAgICAgZmlsZW5hbWUsCiAgICAgICAgICAgIGZsYWdzPXJlLklHTk9SRUNBU0UsCiAgICAgICAgKQogICAgICAgIGlmIHNwbGl0OgogICAgICAgICAgICBzdGVtID0gc3BsaXQuZ3JvdXAoInN0ZW0iKS5sb3dlcigpCiAgICAgICAgICAgIGdyb3VwID0gZ3JvdXBzLnNldGRlZmF1bHQoCiAgICAgICAgICAgICAgICBzdGVtLAogICAgICAgICAgICAgICAgeyJmaWxlcyI6IFtdLCAiY291bnQiOiBpbnQoc3BsaXQuZ3JvdXAoImNvdW50IikpLCAiaW5kaWNlcyI6IFtdfSwKICAgICAgICAgICAgKQogICAgICAgICAgICBpZiBncm91cFsiY291bnQiXSAhPSBpbnQoc3BsaXQuZ3JvdXAoImNvdW50IikpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZ3JvdXBbImZpbGVzIl0uYXBwZW5kKChpbnQoc3BsaXQuZ3JvdXAoImluZGV4IikpLCBmaWxlbmFtZSwgc2l6ZSkpCiAgICAgICAgICAgIGdyb3VwWyJpbmRpY2VzIl0uYXBwZW5kKGludChzcGxpdC5ncm91cCgiaW5kZXgiKSkpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZ3JvdXBzW2ZpbGVuYW1lLmxvd2VyKCldID0gewogICAgICAgICAgICAgICAgImZpbGVzIjogWygxLCBmaWxlbmFtZSwgc2l6ZSldLAogICAgICAgICAgICAgICAgImNvdW50IjogMSwKICAgICAgICAgICAgICAgICJpbmRpY2VzIjogWzFdLAogICAgICAgICAgICB9CgogICAgdmFsaWQ6IGxpc3RbdHVwbGVbbGlzdFtzdHJdLCBpbnRdXSA9IFtdCiAgICBmb3IgZ3JvdXAgaW4gZ3JvdXBzLnZhbHVlcygpOgogICAgICAgIGV4cGVjdGVkID0gZ3JvdXBbImNvdW50Il0KICAgICAgICBmaWxlc193aXRoX3NpemUgPSBncm91cFsiZmlsZXMiXQogICAgICAgIGluZGljZXMgPSBncm91cFsiaW5kaWNlcyJdCiAgICAgICAgaWYgbGVuKGZpbGVzX3dpdGhfc2l6ZSkgIT0gZXhwZWN0ZWQgb3Igc29ydGVkKGluZGljZXMpICE9IGxpc3QoCiAgICAgICAgICAgIHJhbmdlKDEsIGV4cGVjdGVkICsgMSkKICAgICAgICApOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZpbGVzX3dpdGhfc2l6ZS5zb3J0KGtleT1sYW1iZGEgaXRlbTogaXRlbVswXSkKICAgICAgICBmaWxlbmFtZXMgPSBbaXRlbVsxXSBmb3IgaXRlbSBpbiBmaWxlc193aXRoX3NpemVdCiAgICAgICAgc2l6ZSA9IHN1bShpdGVtWzJdIGZvciBpdGVtIGluIGZpbGVzX3dpdGhfc2l6ZSkKICAgICAgICB2YWxpZC5hcHBlbmQoKGZpbGVuYW1lcywgc2l6ZSkpCiAgICByZXR1cm4gdmFsaWQKCgpkZWYgc2VsZWN0X3F1YW50KAogICAgcmVwb19pZDogc3RyLAogICAgc2libGluZ3M6IGxpc3RbQW55XSwKICAgICosCiAgICBtYXhfbW9kZWxfZ2liOiBmbG9hdCA9IERFRkFVTFRfTUFYX01PREVMX0dJQiwKKSAtPiBNb2RlbFNlbGVjdGlvbjoKICAgICIiIlBpY2sgdGhlIHByZWZlcnJlZCBjb21wbGV0ZSBxdWFudCB0aGF0IGZpdHMgdW5kZXIgdGhlIHdlaWdodCBidWRnZXQuIiIiCiAgICBpZiBtYXhfbW9kZWxfZ2liIDw9IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibWF4X21vZGVsX2dpYiBtdXN0IGJlIHBvc2l0aXZlLiIpCiAgICBtYXhfYnl0ZXMgPSBpbnQobWF4X21vZGVsX2dpYiAqIEdJQikKICAgIGZvciBxdWFudCBpbiBRVUFOVF9PUkRFUjoKICAgICAgICBjYW5kaWRhdGVzID0gWwogICAgICAgICAgICBncm91cAogICAgICAgICAgICBmb3IgZ3JvdXAgaW4gX2dyb3Vwc19mb3JfcXVhbnQoc2libGluZ3MsIHF1YW50KQogICAgICAgICAgICBpZiBncm91cFsxXSA8PSBtYXhfYnl0ZXMKICAgICAgICBdCiAgICAgICAgaWYgY2FuZGlkYXRlczoKICAgICAgICAgICAgY2FuZGlkYXRlcy5zb3J0KAogICAgICAgICAgICAgICAga2V5PWxhbWJkYSBncm91cDogKAogICAgICAgICAgICAgICAgICAgIGFueSgibm9tdHAiIGluIG5hbWUubG93ZXIoKSBmb3IgbmFtZSBpbiBncm91cFswXSksCiAgICAgICAgICAgICAgICAgICAgZ3JvdXBbMV0sCiAgICAgICAgICAgICAgICAgICAgZ3JvdXBbMF1bMF0ubG93ZXIoKSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgICAgICAgICBmaWxlbmFtZXMsIHRvdGFsX3NpemUgPSBjYW5kaWRhdGVzWzBdCiAgICAgICAgICAgIHJldHVybiBNb2RlbFNlbGVjdGlvbigKICAgICAgICAgICAgICAgIHJlcG9faWQ9cmVwb19pZCwKICAgICAgICAgICAgICAgIHF1YW50PXF1YW50LAogICAgICAgICAgICAgICAgZmlsZXM9ZmlsZW5hbWVzLAogICAgICAgICAgICAgICAgcHJpbWFyeV9maWxlPWZpbGVuYW1lc1swXSwKICAgICAgICAgICAgICAgIHRvdGFsX3NpemVfYnl0ZXM9dG90YWxfc2l6ZSwKICAgICAgICAgICAgICAgIG1heF9tb2RlbF9naWI9bWF4X21vZGVsX2dpYiwKICAgICAgICAgICAgKQogICAgcmFpc2UgTW9kZWxTZWxlY3Rpb25FcnJvcigKICAgICAgICAiTm8gY29tcGxldGUgUTRfS19NLCBJUTRfWFMsIG9yIFEzX0tfTSBHR1VGIGZpdHMgdGhlICIKICAgICAgICBmInttYXhfbW9kZWxfZ2liOmd9IEdpQiB3ZWlnaHQgYnVkZ2V0IGluIHtyZXBvX2lkfS4iCiAgICApCgoKZGVmIF9oYXNfZ2d1Zl9maWxlcyhhcGk6IEFueSwgcmVwb19pZDogc3RyLCB0b2tlbjogc3RyIHwgTm9uZSkgLT4gYm9vbDoKICAgICIiIkNoZWNrIGEgc2VhcmNoIHJlc3VsdCBmb3IgR0dVRiBmaWxlcyB3aXRob3V0IGRvd25sb2FkaW5nIG1vZGVsIGRhdGEuIiIiCiAgICB0cnk6CiAgICAgICAgaW5mbyA9IGFwaS5tb2RlbF9pbmZvKHJlcG9faWQsIGZpbGVzX21ldGFkYXRhPUZhbHNlLCB0b2tlbj10b2tlbikKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICByZXR1cm4gYW55KAogICAgICAgIHN0cihfZ2V0KGl0ZW0sICJyZmlsZW5hbWUiLCAiIikpLmxvd2VyKCkuZW5kc3dpdGgoIi5nZ3VmIikKICAgICAgICBmb3IgaXRlbSBpbiAoX2dldChpbmZvLCAic2libGluZ3MiLCBbXSkgb3IgW10pCiAgICApCgoKZGVmIGRpc2NvdmVyX2NhbmRpZGF0ZV9yZXBvcyhhcGk6IEFueSwgdG9rZW46IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBsaXN0W3N0cl06CiAgICAiIiJGaW5kIG90aGVyIHNpbWlsYXJseSBuYW1lZCB1bmNlbnNvcmVkIEdHVUYgcmVwb3NpdG9yaWVzIGZvciB1c2VyIGNob2ljZS4iIiIKICAgIHRyeToKICAgICAgICBtb2RlbHMgPSBhcGkubGlzdF9tb2RlbHMoc2VhcmNoPSJRd2VuMy44LTI3QiIsIGxpbWl0PTEwMCwgdG9rZW49dG9rZW4pCiAgICBleGNlcHQgVHlwZUVycm9yOgogICAgICAgIG1vZGVscyA9IGFwaS5saXN0X21vZGVscyhzZWFyY2g9IlF3ZW4zLjgtMjdCIikKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIFtdCgogICAgY2FuZGlkYXRlczogbGlzdFtzdHJdID0gW10KICAgIGN1ZXMgPSAoInVuY2Vuc29yZWQiLCAidW5sZWFzaGVkIiwgImFibGl0ZXJhdCIsICJoZXJldGljIikKICAgIGZvciBtb2RlbCBpbiBtb2RlbHM6CiAgICAgICAgcmVwb19pZCA9IHN0cihfZ2V0KG1vZGVsLCAiaWQiLCAiIikgb3IgX2dldChtb2RlbCwgIm1vZGVsSWQiLCAiIikpCiAgICAgICAgbG93ZXJlZCA9IHJlcG9faWQubG93ZXIoKQogICAgICAgIGlmIG5vdCByZXBvX2lkIG9yIG5vdCBhbnkoY3VlIGluIGxvd2VyZWQgZm9yIGN1ZSBpbiBjdWVzKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBfaGFzX2dndWZfZmlsZXMoYXBpLCByZXBvX2lkLCB0b2tlbik6CiAgICAgICAgICAgIGNhbmRpZGF0ZXMuYXBwZW5kKHJlcG9faWQpCiAgICAgICAgaWYgbGVuKGNhbmRpZGF0ZXMpID49IDEyOgogICAgICAgICAgICBicmVhawogICAgcmV0dXJuIHNvcnRlZChzZXQoY2FuZGlkYXRlcyksIGtleT1zdHIubG93ZXIpCgoKZGVmIHJlc29sdmVfbW9kZWwoCiAgICBhcGk6IEFueSwKICAgIHJlcG9faWQ6IHN0ciwKICAgICosCiAgICB0b2tlbjogc3RyIHwgTm9uZSA9IE5vbmUsCiAgICBtYXhfbW9kZWxfZ2liOiBmbG9hdCA9IERFRkFVTFRfTUFYX01PREVMX0dJQiwKICAgIG1pc3NpbmdfcmVwb19lcnJvcnM6IHR1cGxlW3R5cGVbQmFzZUV4Y2VwdGlvbl0sIC4uLl0gPSAoKSwKKSAtPiBNb2RlbFNlbGVjdGlvbjoKICAgICIiIlZlcmlmeSBhIHJlcG9zaXRvcnksIG9yIHNob3cgc2VhcmNoIHJlc3VsdHMgaWYgaXQgaXMgbWlzc2luZy4iIiIKICAgIHRyeToKICAgICAgICBpbmZvID0gYXBpLm1vZGVsX2luZm8ocmVwb19pZCwgZmlsZXNfbWV0YWRhdGE9VHJ1ZSwgdG9rZW49dG9rZW4pCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBpZiBub3QgbWlzc2luZ19yZXBvX2Vycm9ycyBvciBub3QgaXNpbnN0YW5jZShleGMsIG1pc3NpbmdfcmVwb19lcnJvcnMpOgogICAgICAgICAgICByYWlzZQogICAgICAgIGNhbmRpZGF0ZXMgPSBkaXNjb3Zlcl9jYW5kaWRhdGVfcmVwb3MoYXBpLCB0b2tlbikKICAgICAgICBzdWZmaXggPSAiXG4iLmpvaW4oZiIgIHtjYW5kaWRhdGV9IiBmb3IgY2FuZGlkYXRlIGluIGNhbmRpZGF0ZXMpCiAgICAgICAgZ3VpZGFuY2UgPSAoCiAgICAgICAgICAgICJTZXQgTU9ERUxfUkVQTyBpbiB0aGUgS2FnZ2xlIG5vdGVib29rIHRvIG9uZSBvZiB0aGVzZSByZXBvc2l0b3JpZXMsICIKICAgICAgICAgICAgInRoZW4gcmVydW4gdGhlIHNlcnZlciBjZWxsLiIKICAgICAgICApCiAgICAgICAgaWYgc3VmZml4OgogICAgICAgICAgICBndWlkYW5jZSA9IGYie2d1aWRhbmNlfVxuQ2FuZGlkYXRlczpcbntzdWZmaXh9IgogICAgICAgIHJhaXNlIE1vZGVsU2VsZWN0aW9uRXJyb3IoCiAgICAgICAgICAgIGYiUmVxdWVzdGVkIEh1Z2dpbmcgRmFjZSByZXBvc2l0b3J5IHdhcyBub3QgZm91bmQ6IHtyZXBvX2lkfS4ge2d1aWRhbmNlfSIKICAgICAgICApIGZyb20gZXhjCgogICAgc2libGluZ3MgPSBsaXN0KF9nZXQoaW5mbywgInNpYmxpbmdzIiwgW10pIG9yIFtdKQogICAgaWYgbm90IHNpYmxpbmdzOgogICAgICAgIHJhaXNlIE1vZGVsU2VsZWN0aW9uRXJyb3IoZiJSZXBvc2l0b3J5IHtyZXBvX2lkfSBjb250YWlucyBubyBmaWxlcy4iKQogICAgcmV0dXJuIHNlbGVjdF9xdWFudChyZXBvX2lkLCBzaWJsaW5ncywgbWF4X21vZGVsX2dpYj1tYXhfbW9kZWxfZ2liKQoKCmRlZiBtYWluKGFyZ3Y6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lKSAtPiBpbnQ6CiAgICAiIiJSZXNvbHZlIGEgcmVwb3NpdG9yeSB0aHJvdWdoIHRoZSBIdWdnaW5nIEZhY2UgQVBJIGFuZCBwcmludCBpdHMgY2hvaWNlLiIiIgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVwbyIsIGRlZmF1bHQ9b3MuZW52aXJvbi5nZXQoIk1PREVMX1JFUE8iLCBERUZBVUxUX1JFUE8pKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tYXgtbW9kZWwtZ2liIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD1ERUZBVUxUX01BWF9NT0RFTF9HSUIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLWpzb24iLCBhY3Rpb249InN0b3JlX3RydWUiLCBoZWxwPSJQcmludCBtYWNoaW5lLXJlYWRhYmxlIEpTT04uIgogICAgKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKGFyZ3YpCgogICAgdHJ5OgogICAgICAgIGZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBIZkFwaQogICAgICAgIGZyb20gaHVnZ2luZ2ZhY2VfaHViLnV0aWxzIGltcG9ydCBSZXBvc2l0b3J5Tm90Rm91bmRFcnJvcgogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHByaW50KAogICAgICAgICAgICAiaHVnZ2luZ2ZhY2VfaHViIGlzIHJlcXVpcmVkIGluIHRoZSBLYWdnbGUgc2VydmVyIGVudmlyb25tZW50LiIsCiAgICAgICAgICAgIGZpbGU9c3lzLnN0ZGVyciwKICAgICAgICApCiAgICAgICAgcmV0dXJuIDIKCiAgICB0b2tlbiA9IG9zLmVudmlyb24uZ2V0KCJIRl9UT0tFTiIpIG9yIE5vbmUKICAgIHRyeToKICAgICAgICBzZWxlY3Rpb24gPSByZXNvbHZlX21vZGVsKAogICAgICAgICAgICBIZkFwaSgpLAogICAgICAgICAgICBhcmdzLnJlcG8sCiAgICAgICAgICAgIHRva2VuPXRva2VuLAogICAgICAgICAgICBtYXhfbW9kZWxfZ2liPWFyZ3MubWF4X21vZGVsX2dpYiwKICAgICAgICAgICAgbWlzc2luZ19yZXBvX2Vycm9ycz0oUmVwb3NpdG9yeU5vdEZvdW5kRXJyb3IsKSwKICAgICAgICApCiAgICBleGNlcHQgTW9kZWxTZWxlY3Rpb25FcnJvciBhcyBleGM6CiAgICAgICAgcHJpbnQoc3RyKGV4YyksIGZpbGU9c3lzLnN0ZGVycikKICAgICAgICByZXR1cm4gMgogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiSHVnZ2luZyBGYWNlIGxvb2t1cCBmYWlsZWQgKHt0eXBlKGV4YykuX19uYW1lX199KTsgY2hlY2sgS2FnZ2xlIEludGVybmV0ICIKICAgICAgICAgICAgImFjY2VzcyBhbmQgcmVwb3NpdG9yeSBwZXJtaXNzaW9ucy4iLAogICAgICAgICAgICBmaWxlPXN5cy5zdGRlcnIsCiAgICAgICAgKQogICAgICAgIHJldHVybiAxCgogICAgcmVzdWx0ID0gYXNkaWN0KHNlbGVjdGlvbikKICAgIGlmIGFyZ3MuanNvbjoKICAgICAgICBwcmludChqc29uLmR1bXBzKHJlc3VsdCkpCiAgICBlbHNlOgogICAgICAgIHByaW50KGYiUmVwb3NpdG9yeToge3NlbGVjdGlvbi5yZXBvX2lkfSIpCiAgICAgICAgcHJpbnQoZiJRdWFudDoge3NlbGVjdGlvbi5xdWFudH0iKQogICAgICAgIHByaW50KGYiVG90YWwgbW9kZWwgd2VpZ2h0IHNpemU6IHtzZWxlY3Rpb24udG90YWxfc2l6ZV9ieXRlcyAvIEdJQjouMmZ9IEdpQiIpCiAgICAgICAgcHJpbnQoZiJQcmltYXJ5IEdHVUY6IHtzZWxlY3Rpb24ucHJpbWFyeV9maWxlfSIpCiAgICAgICAgcHJpbnQoIkZpbGVzIHRvIGRvd25sb2FkOiIpCiAgICAgICAgZm9yIGZpbGVuYW1lIGluIHNlbGVjdGlvbi5maWxlczoKICAgICAgICAgICAgcHJpbnQoZiIgIHtmaWxlbmFtZX0iKQogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg=="
}
for filename, content in embedded_files.items():
    file_path = server_dir / filename
    file_path.write_bytes(base64.b64decode(content))
    if filename.endswith('.sh'):
        file_path.chmod(0o700)
print(f"Prepared server scripts in {server_dir}")


In [ ]:
%cd /kaggle/working/qwenfree_server
!bash bootstrap.sh
